# Preprocessing

Preprocessing for reproducing the Deep ConvNet results in Table 2 of Schirrmeister et al. (2017).

Section references are to the [arXiv version](https://arxiv.org/abs/1703.05051v5) of the paper, which includes the supplementary material as Appendix A.

| Step | BCIC IV 2a | HGD | Paper |
|---|---|---|---|
| Channels | 22 EEG (drop EOG, stim) | 44 sensors over the motor cortex | A.6, A.7 (p.9) and  |
| Units | V → µV | V → µV | (unit of the A.7 threshold) |
| Resampling | — (already 250 Hz) | 500 → 250 Hz | A.7 (p.10) |
| Bad trials | trials exceeding ±800 µV are dropped | same | A.7 (p.10) |
| Filter, 0 Hz rows | low-pass 38 Hz | none (0–125 Hz = up to Nyquist) | A.7 (p.10) |
| Filter, 4 Hz rows | band-pass 4–38 Hz | high-pass 4 Hz | 2.7.1 (p.5405/15), A.7 (p.10) |
| Filter type | causal 3rd order Butterworth | same | 2.7.1 (p.5405/15) |
| Standardization | exponential moving standardization | same | A.7, Eqs. 4–6 (p.10) |
| Trial window | −0.5 s to 4 s after the cue | same | 2.5.3 (p.5400/p.10) |
| Train/test split | session 1 / session 2 | all runs but the last 2 / last 2 runs | A.6 (p.9) |

The paper does not state the order of cleaning, resampling and filtering. Here cleaning is done on the unfiltered µV data.

Datasets are loaded with `data_loading.py` (MOABB, cached in `datasets/`).

In [14]:
import mne
import numpy as np
from braindecode.preprocessing import (
    Preprocessor,
    create_windows_from_events,
    exponential_moving_standardize,
    preprocess,
)

from data_loading import load_dataset, split_train_test

mne.set_log_level("ERROR")  # hide MNE's per-channel loading messages

## Settings

Each setting is quoted from the paper:

- **HGD channels**, A.7: *"for the High-Gamma Dataset, we used only those sensors covering the motor cortex: all central electrodes (45), except the Cz electrode which served as the recording reference electrode."*
- **Low-pass**, A.7: *"we low-pass filtered the BCI Competition Dataset to below 38 Hz."* HGD gets no low-pass, so f_end is the 125 Hz Nyquist frequency after resampling.
- **Resampling**, A.7: *"We resampled the High-Gamma Dataset to 250 Hz, i.e., the same as the BCI Competition Dataset."*
- **Filter**, 2.7.1: *"Filtering was done with a causal 3rd order Butterworth filter."* In MNE, `phase="forward"` applies the IIR filter once, forward in time, which makes it causal.
- **Cleaning**, A.7: *"removed trials where at least one channel had a value outside ±800 µV."*
- **Standardization**, A.7: *"electrode-wise exponential moving standardization with a decay factor of 0.999"*, initialized with *"the mean and the variance of the first 1000 samples."*
- **Trial window**, 2.5.3: *"4.5-second trials (from 500 ms before trial start cue until trial end cue …)"*
- **Classes**, A.6: left hand, right hand, feet and tongue (BCIC IV 2a); left hand, right hand, feet and rest (HGD).

In [15]:
HGD_MOTOR_CHANNELS = [
    "FC5", "FC1", "FC2", "FC6", "C3", "C4", "CP5", "CP1", "CP2", "CP6",
    "FC3", "FCz", "FC4", "C5", "C1", "C2", "C6", "CP3", "CPz", "CP4",
    "FFC5h", "FFC3h", "FFC4h", "FFC6h", "FCC5h", "FCC3h", "FCC4h", "FCC6h",
    "CCP5h", "CCP3h", "CCP4h", "CCP6h", "CPP5h", "CPP3h", "CPP4h", "CPP6h",
    "FFC1h", "FFC2h", "FCC1h", "FCC2h", "CCP1h", "CCP2h", "CPP1h", "CPP2h",
]
HIGH_CUT_HZ = {"BCIC_IV_2a": 38.0, "HGD": None}  # None: no low-pass
SFREQ = 250.0
FILTER_PARAMS = dict(
    method="iir", iir_params=dict(order=3, ftype="butter"), phase="forward"
)
BAD_TRIAL_THRESHOLD_UV = 800.0
FACTOR_NEW = 1e-3  # 1 - decay factor of 0.999
INIT_BLOCK_SIZE = 1000
TRIAL_START_OFFSET_S = -0.5
CLASS_MAPPING = {
    "BCIC_IV_2a": {"left_hand": 0, "right_hand": 1, "feet": 2, "tongue": 3},
    "HGD": {"right_hand": 0, "left_hand": 1, "rest": 2, "feet": 3},
}
BAD_TRIAL = "bad_trial"

## Bad trial rejection (A.7)

*"We also only minimally cleaned the data sets to remove extreme high-amplitude recording artifacts. Our cleaning method thus only removed trials where at least one channel had a value outside ±800 µV."*

Applied to both datasets. Trials above the threshold are relabelled as `bad_trial`. This runs before filtering and standardization, while the data is still in µV. Relabelled trials are not in `CLASS_MAPPING`, so windowing skips them.

In [16]:
def mark_high_amplitude_trials(raw, threshold_uv=BAD_TRIAL_THRESHOLD_UV):
    data = raw.get_data()
    sfreq = raw.info["sfreq"]
    descriptions = raw.annotations.description.copy()
    for i, (onset, duration) in enumerate(
        zip(raw.annotations.onset, raw.annotations.duration)
    ):
        start = int(round((onset - raw.first_time) * sfreq))
        stop = start + int(round(duration * sfreq))
        if np.abs(data[:, start:stop]).max() > threshold_uv:
            descriptions[i] = BAD_TRIAL
    raw.annotations.description = descriptions
    return raw

## Preprocessing pipeline (2.7.1, A.7)

`low_cut_hz` is `4.0` for the 4–f_end rows of Table 2 and `None` for the 0–f_end rows (2.7.1).

For BCIC IV 2a, the 4 Hz high-pass and the 38 Hz low-pass are a single band-pass filter. A.7: *"for the BCI Competition Dataset, we bandpass-filtered to 4-38 Hz, so the previous lowpass-filter step was merged with the highpass-filter step."*

Exponential moving standardization (A.7, Eqs. 4–6):

$$x'_t = (x_t - \mu_t) / \sqrt{\sigma_t^2}, \quad \mu_t = 0.001 x_t + 0.999 \mu_{t-1}, \quad \sigma_t^2 = 0.001 (x_t - \mu_t)^2 + 0.999 \sigma_{t-1}^2$$

In [17]:
def preprocess_dataset(dataset, name, low_cut_hz, n_jobs=1):
    low_cut_hz = low_cut_hz or None
    high_cut_hz = HIGH_CUT_HZ[name]

    if name == "HGD":
        preprocessors = [
            Preprocessor("pick", picks=HGD_MOTOR_CHANNELS),  # A.7
            Preprocessor(lambda data: np.multiply(data, 1e6)),  # V -> uV
            Preprocessor("resample", sfreq=SFREQ),  # A.7
        ]
    else:
        preprocessors = [
            # 22 EEG electrodes (A.6), dropping the EOG and stim channels
            Preprocessor("pick_types", eeg=True, meg=False, stim=False),
            Preprocessor(lambda data: np.multiply(data, 1e6)),  # V -> uV
        ]
    preprocessors.append(  # A.7
        Preprocessor(mark_high_amplitude_trials, apply_on_array=False)
    )
    if low_cut_hz is not None or high_cut_hz is not None:  # 2.7.1, A.7
        preprocessors.append(
            Preprocessor(
                "filter", l_freq=low_cut_hz, h_freq=high_cut_hz, **FILTER_PARAMS
            )
        )
    preprocessors.append(  # A.7, Eqs. 4-6
        Preprocessor(
            exponential_moving_standardize,
            factor_new=FACTOR_NEW,
            init_block_size=INIT_BLOCK_SIZE,
        )
    )
    preprocess(dataset, preprocessors, n_jobs=n_jobs)
    return dataset

## Trial windows (2.5.3)

*"for both datasets we had 4.5-second trials (from 500 ms before trial start cue until trial end cue, as that worked best in preliminary experiments)"*

At 250 Hz that is 1125 samples. Extra keyword arguments, e.g. `window_size_samples` and `window_stride_samples` for cropped training (2.5.4), are passed on to `create_windows_from_events`.

In [18]:
def create_trial_windows(dataset, name, trial_start_offset_s=TRIAL_START_OFFSET_S,
                         **kwargs):
    sfreq = dataset.datasets[0].raw.info["sfreq"]
    assert all(ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets)
    return create_windows_from_events(
        dataset,
        trial_start_offset_samples=int(trial_start_offset_s * sfreq),
        trial_stop_offset_samples=0,
        mapping=CLASS_MAPPING[name],
        preload=True,
        **kwargs,
    )

## Check on subject 1

Run all four configurations and check channels, sampling rate, filter band, trial counts and that the data is roughly standardized (mean ≈ 0, std ≈ 1).

The train/test split follows A.6: BCIC IV 2a trains on session 1 and tests on session 2 (288 trials each); HGD tests on the last 2 runs (approx. 160 trials).

In [19]:
subject_ids = [1]

for name in ["BCIC_IV_2a", "HGD"]:
    for low_cut_hz in [None, 4.0]:
        ds = preprocess_dataset(load_dataset(name, subject_ids), name, low_cut_hz)
        raw = ds.datasets[0].raw
        train_set, test_set = split_train_test(create_trial_windows(ds, name))
        X, _, _ = train_set[0]
        y_train = np.array([train_set[i][1] for i in range(len(train_set))])
        print(
            f"{name}, low cut {low_cut_hz}: {len(raw.ch_names)} channels, "
            f"{raw.info['sfreq']:.0f} Hz, band {raw.info['highpass']:.0f}-{raw.info['lowpass']:.0f} Hz, "
            f"trials train/test {len(train_set)}/{len(test_set)}, window {X.shape}, "
            f"class counts {np.bincount(y_train)}, mean {X.mean():.2f}, std {X.std():.2f}"
        )

KeyboardInterrupt: 

## Check bad trial rejection

Count trials above the threshold directly in the µV data and compare with what `mark_high_amplitude_trials` marks.

In [ ]:
for name, subject in [("BCIC_IV_2a", 1), ("HGD", 1), ("HGD", 2)]:
    for ds in load_dataset(name, [subject]).datasets:
        raw = ds.raw
        if name == "HGD":
            raw.pick(HGD_MOTOR_CHANNELS).resample(SFREQ)
        else:
            raw.pick_types(eeg=True)
        raw.apply_function(lambda data: data * 1e6)
        data, sfreq = raw.get_data(), raw.info["sfreq"]
        trial_max = [
            np.abs(data[:, int(round((onset - raw.first_time) * sfreq)):][:, :int(round(duration * sfreq))]).max()
            for onset, duration in zip(raw.annotations.onset, raw.annotations.duration)
        ]
        n_marked = (mark_high_amplitude_trials(raw).annotations.description == BAD_TRIAL).sum()
        print(
            f"{name} subject {subject}, {ds.description['session']}/{ds.description['run']}: "
            f"{len(trial_max)} trials, max {max(trial_max):.0f} uV, "
            f"above 800 uV: {sum(m > BAD_TRIAL_THRESHOLD_UV for m in trial_max)}, marked: {n_marked}"
        )

BCIC_IV_2a subject 1, 0train/0: 48 trials, max 71 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 0train/1: 48 trials, max 100 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 0train/2: 48 trials, max 78 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 0train/3: 48 trials, max 62 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 0train/4: 48 trials, max 77 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 0train/5: 48 trials, max 80 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 1test/0: 48 trials, max 73 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 1test/1: 48 trials, max 100 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 1test/2: 48 trials, max 100 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 1test/3: 48 trials, max 60 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 1test/4: 48 trials, max 60 uV, above 800 uV: 0, marked: 0
BCIC_IV_2a subject 1, 1test/5: 48 trials, max 58 uV, above 800 uV: 0, marked: 0


No stim channel nor annotations found, skipping setting annotations.
No stim channel nor annotations found, skipping setting annotations.


HGD subject 1, 0/0train: 320 trials, max 541 uV, above 800 uV: 0, marked: 0
HGD subject 1, 0/1test: 160 trials, max 549 uV, above 800 uV: 0, marked: 0


No stim channel nor annotations found, skipping setting annotations.
No stim channel nor annotations found, skipping setting annotations.


HGD subject 2, 0/0train: 813 trials, max 1728 uV, above 800 uV: 2, marked: 2
HGD subject 2, 0/1test: 160 trials, max 268 uV, above 800 uV: 0, marked: 0
